In [73]:
import pandas as pd
import numpy as np


def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

def load_atributes(dataset: pd.DataFrame, date: pd.Timestamp, years_limit: int = 1, matches_limit: int = 5) -> pd.DataFrame:

    def get_historial(date, team: str) -> pd.DataFrame:

        #filtramos el dataset por el equipo y la fecha
        df = dataset[(dataset["home"] == team) | (dataset["away"] == team)]
        df = df[(date - pd.DateOffset(years=years_limit) < df["date"]) & (df["date"] < date)]

        #eliminamos los que sean anteriores a un limite 
        return df

    def get_wins(dataset: pd.DataFrame, team: str) -> int:
        #contamos las victorias del equipo
        return len(dataset[(dataset["home"] == team) & (dataset["result"] == "L")]) + len(dataset[(dataset["away"] == team) & (dataset["result"] == "V")])

    def comparar(win_reate_team1: float, win_reate_team2: float) -> str:
        #comparamos las victorias de los equipos
        if win_reate_team1 > win_reate_team2:
            return "L"
        elif win_reate_team1 < win_reate_team2:
            return "V"
        else:
            return "E"

    def get_condition(date, team: str) -> int:
        #obtenemos el historial del equipo
        df = get_historial(date, team)

        #me quedo con los ultimos partidos del equipo (si los hay)
        df = df.tail(matches_limit)

        #TODO : quizas se puede hacer por puntos en vez de cantidad de victorias, pero por ahora lo dejamos asi
        return (get_wins(df, team) / len(df) if len(df) > 0 else 0)
    

    #creamos las nuevas columnas con el historial historico de los equipos
    df["historial"] = comparar(df["home"].apply(get_historial, args=(df["date"], df["home"])), df["away"].apply(get_historial, args=(df["date"], df["away"])))

    #creamos las nuevas columnas con las condiciones recientes de los equipos
    df["condition_match"] = comparar(df["home"].apply(get_condition, args=(df["date"], df["home"])), df["away"].apply(get_condition, args=(df["date"], df["away"])))  
    
    
    get_condition_team1 = get_condition(df["date"].iloc[0], "CA Penarol")
    get_condition_team2 = get_condition(df["date"].iloc[0],  "Nacional")

    print(f"Condicion de CA Penarol: {get_condition_team1}")
    print(f"Condicion de Nacional: {get_condition_team2}")

    return comparar(get_condition_team1, get_condition_team2)

    







In [74]:
df = load_dataset("futbol_uruguayo.csv")

print(df.head())

print (load_atributes(df, pd.Timestamp("2025-01-01"), years_limit=1, matches_limit=5))

                   home                              away       date  gh  ga  \
0           Bella Vista                 Defensor Sporting 1932-03-05   1   2   
1            CA Penarol                       River Plate 1932-03-05   1   1   
2       Central Espanol        Rampla Juniors Futbol Club 1932-03-05   1   0   
3  Montevideo Wanderers                       Racing Club 1932-03-05   3   0   
4              Nacional  Institucion Atletica Sud America 1932-03-05   2   0   

  result  
0      V  
1      E  
2      L  
3      L  
4      L  


TypeError: get_historial() takes 2 positional arguments but 3 were given